# 🧪 Checkpoint 1 — Data & Geographic Assessment & Spatial Integration
## Project: Mérida Urban Intelligence | Team: Sunchos Team

### Objectives:
1. Inventory and profile raw data sources (Demographics, DENUE, AGEB Cartography, Crime).
2. Document geographic-unit decision (AGEB) and justification.
3. Demonstrate Point-to-Polygon spatial integration (Spatial Join).
4. List source variables required by analytical KPIs.

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point, Polygon
from src.etl.config import CRS_WGS84, CRS_METRIC_MEXICO, DATA_RAW_DIR
from src.etl.extract import list_raw_files
from src.etl.transform import standardize_ageb_geometries, spatial_join_points_to_polygons

print("✅ Libraries imported successfully.")

## 1. Data Source Inventory & Profiling
Scanning `data/raw/` for datasets:

In [ ]:
inventory = list_raw_files()
for category, files in inventory.items():
    print(f"📂 Category: {category.upper()} -> {len(files)} files found.")
    for f in files:
        print(f"   - {f.name} ({f.stat().st_size / 1024:.1f} KB)")

## 2. Geographic Unit Decision: AGEB Urbana de Mérida

### Justification:
- **Balance of Resolution**: Avoids statistical suppression (secreto estadístico) present at the Manzana (block) level while maintaining fine-grained intra-urban spatial granularity.
- **Source Compatibility**: Matches official INEGI Censo 2020 aggregations and DENUE coordinates.
- **Spatial Metric Accuracy**: Reprojected to EPSG:6372 / EPSG:32616 for precise area calculations ($km^2$).

## 3. Point-to-Polygon Spatial Integration Test
Testing spatial join between simulated/sample point coordinates and AGEB polygons:

In [ ]:
# Create sample polygons (Mérida AGEBs mock/test)
poly1 = Polygon([(-89.65, 20.95), (-89.60, 20.95), (-89.60, 21.00), (-89.65, 21.00)])
poly2 = Polygon([(-89.60, 20.95), (-89.55, 20.95), (-89.55, 21.00), (-89.60, 21.00)])

gdf_sample_polys = gpd.GeoDataFrame(
    {
        'cvegeo': ['310500001001A', '310500001002A'],
        'nom_asentamiento': ['Centro', 'Itzimná'],
        'geometry': [poly1, poly2]
    },
    crs=CRS_WGS84
)

# Sample points (Incidents & Businesses)
df_sample_points = pd.DataFrame({
    'id': [101, 102, 103],
    'tipo': ['Comercio', 'Delito', 'Servicio'],
    'latitud': [20.97, 20.98, 20.96],
    'longitud': [-89.63, -89.58, -89.62]
})

# Execute Spatial Join
joined_res = spatial_join_points_to_polygons(df_sample_points, gdf_sample_polys)
print("✅ Spatial Join Test Output:")
print(joined_res[['id', 'tipo', 'cvegeo', 'latitud', 'longitud']])